# 03 Export, Inference & Release Checks
Objective: Verify artifact reloading, measure CPU inference latency, perform ONNX export/parity tests where applicable, and generate unified manifests for the FastAPI backend and React frontend.

In [2]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from pathlib import Path
import time
import json
import numpy as np
from google.colab import drive

# Mount Google Drive to ensure local checkpoint paths exist
drive.mount('/content/drive', force_remount=True)

ROOT = Path('/content/drive/MyDrive/NLP_Lab')
AMAZON_CKPT = ROOT / 'checkpoints/word2vec_bilstm'
GOEMOTIONS_CKPT = ROOT / 'checkpoints/distilbert_goemotions'

print("--- 1. Testing Model B (DistilBERT GoEmotions) Reload ---")
# Convert Path objects to string representations for Hugging Face loader functions
tokenizer_b = AutoTokenizer.from_pretrained(str(GOEMOTIONS_CKPT))
model_b = AutoModelForSequenceClassification.from_pretrained(str(GOEMOTIONS_CKPT))
model_b.eval()

with open(GOEMOTIONS_CKPT / 'thresholds.json') as f:
    thresholds = json.load(f)
threshold = thresholds['global_threshold']

sample_text = "I am extremely excited and grateful for this amazing opportunity!"
inputs = tokenizer_b(sample_text, return_tensors="pt", padding=True, truncation=True, max_length=128)

start_time = time.time()
with torch.no_grad():
    outputs = model_b(**inputs)
    probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]
latency_b = (time.time() - start_time) * 1000

pred_indices = np.where(probs >= threshold)[0].tolist()
print(f"Inference Latency (DistilBERT CPU): {latency_b:.2f}ms")
print(f"Predicted emotion indices: {pred_indices}")

print("\nSTATUS: PASS. Model B reloaded and verified.")

Mounted at /content/drive
--- 1. Testing Model B (DistilBERT GoEmotions) Reload ---


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Inference Latency (DistilBERT CPU): 489.31ms
Predicted emotion indices: [0, 13, 15]

STATUS: PASS. Model B reloaded and verified.


In [3]:
import hashlib

def compute_sha256(file_path):
    sha256_hash = hashlib.sha256()
    with open(file_path, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

print("--- Generating Unified Model Manifests ---")
manifests_dir = ROOT / 'artifacts/model-manifests'
manifests_dir.mkdir(parents=True, exist_ok=True)

manifest = {
    "project": "Dual-Architecture NLP Classification Platform",
    "models": {
        "amazon_rating_bilstm": {
            "architecture": "Word2Vec Skip-gram + Bidirectional LSTM",
            "checkpoint_path": str(AMAZON_CKPT / 'w2v_bilstm_best.pt'),
            "sha256": compute_sha256(AMAZON_CKPT / 'w2v_bilstm_best.pt'),
            "classes": 5
        },
        "goemotions_distilbert": {
            "architecture": "DistilBERT for Sequence Classification (Multi-Label)",
            "checkpoint_path": str(GOEMOTIONS_CKPT / 'model.safetensors'),
            "sha256": compute_sha256(GOEMOTIONS_CKPT / 'model.safetensors'),
            "classes": 28,
            "optimal_threshold": threshold
        }
    }
}

manifest_path = manifests_dir / 'unified_model_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"Unified manifest saved to {manifest_path}")
print("STATUS: PASS. Ready for FastAPI and backend integration.")

--- Generating Unified Model Manifests ---
Unified manifest saved to /content/drive/MyDrive/NLP_Lab/artifacts/model-manifests/unified_model_manifest.json
STATUS: PASS. Ready for FastAPI and backend integration.
